In [1]:
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import PandasTools

In [42]:
#get data

all_data = 'benchmark_datasets/tox21/tox21_10k_data_all.sdf'
source = PandasTools.LoadSDF(all_data,
                            smilesName='SMILES',
                            molColName='Molecule',
                            includeFingerprints=True)

[16:13:46] The 2 defining bonds for an atropisomer are co-planar - atoms are: 4 10
[16:13:46] Explicit valence for atom # 3 Cl, 2, is greater than permitted
[16:13:46] ERROR: Could not sanitize molecule ending on line 21572
[16:13:46] ERROR: Explicit valence for atom # 3 Cl, 2, is greater than permitted
[16:13:49] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 1 ignored.
[16:13:49] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 1 ignored.
[16:13:49] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 6 ignored.
[16:13:49] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 6 ignored.
[16:13:49] The 2 defining bonds for an atropisomer are co-planar - atoms are: 4 10
[16:13:51] Explicit valence for atom # 2 Si, 8, is greater than permitted
[16:13:51] ERROR: Could not sanitize molecule ending on line 346021
[16:13:51] ERROR: Explicit valence for atom # 

In [43]:
df = source.copy()
df.head(5)

,Formula,FW,DSSTox_CID,SR-HSE,ID,SMILES,Molecule,NR-AR,SR-ARE,NR-Aromatase,NR-ER-LBD,NR-AhR,SR-MMP,NR-ER,NR-PPAR-gamma,SR-p53,SR-ATAD5,NR-AR-LBD
0,C27H25ClN6,468.9806 (35.4535+224.2805+209.2465),25848,0,NCGC00178831-03,C[n+]1c2cc(N)ccc2cc2ccc(N)cc21.Nc1ccc2cc3ccc(N...,<rdkit.Chem.rdchem.Mol object at 0x7f4c9d003220>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C20H6Br4Na2O5,691.8542 (645.8757+22.9892+22.9892),5234,0,NCGC00166114-03,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,<rdkit.Chem.rdchem.Mol object at 0x7f4c9d003840>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C47H83NO17,934.1584 (916.1205+18.0379),28909,0,NCGC00263563-01,CO[C@@H]1[C@@H](OC)[C@H](C)[C@@](O)(CC(=O)[O-]...,<rdkit.Chem.rdchem.Mol object at 0x7f4c9d003a00>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C52H54N4O12,927.0048 (329.4575+89.0275+89.0275+329.4575+90...,5513,1,NCGC00013058-02,CN(C)c1ccc(C(=C2C=CC(=[N+](C)C)C=C2)c2ccccc2)c...,<rdkit.Chem.rdchem.Mol object at 0x7f4c9d003680>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C66H87N17O14,1342.5025 (1282.4505+60.0520),26683,NaN,NCGC00167516-01,CC(=O)O.CCNC(=O)[C@@H]1CCCN1C(=O)[C@H](CCCNC(=...,<rdkit.Chem.rdchem.Mol object at 0x7f4c9d003ca0>,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [46]:
df.dtypes

Formula          object
FW               object
DSSTox_CID       object
SR-HSE           object
ID               object
SMILES           object
Molecule         object
NR-AR            object
SR-ARE           object
NR-Aromatase     object
NR-ER-LBD        object
NR-AhR           object
SR-MMP           object
NR-ER            object
NR-PPAR-gamma    object
SR-p53           object
SR-ATAD5         object
NR-AR-LBD        object
dtype: object

## Steps to create dataset
1. <b>Merge the duplicate smile strings into a single row with following rules:</b>
   1. adding(nan,y) = y -> {0, 1}
   2. adding(x,nan) = x -> {0, 1}
   3. adding(nan,nan) = nan
   4. adding(1,0) = 1
   5. adding(0,1) = 1
   6. adding(x, y) = x if x==y
  
2. <b>If there are rows where all values are nan, remove them</b>
3. Create a map of SMILE string -> MorganFingerprint and store in a pickle file --> <b>Can do this on the fly as the DS is not that big</b>
4. <b>Create dataset in csv format where the first entry is the smile string and there after all the order `['SR-HSE','NR-AR', 'SR-ARE', 'NR-Aromatase', 'NR-ER-LBD', 'NR-AhR', 'SR-MMP',\
       'NR-ER', 'NR-PPAR-gamma', 'SR-p53', 'SR-ATAD5', 'NR-AR-LBD']`</b>
5. <b>Repeat the same for test dataset</b>


In [58]:
df_merged = df.groupby('SMILES').agg({
    col: lambda group: group.apply(merge_all_values) for col in tests
})
df_merged.head()

TypeError: 'float' object is not iterable

In [ ]:

# Reset index if needed
df_merged = df_merged.reset_index()